# SQL Results in NumPy

*Convert ordered SQL measures to arrays and interpret numeric summaries and month-to-month changes.*


## Prepare the Sales Table

The sales table holds five prepared records in an in-memory
SQLite database. `sale_id` is a primary key, and the other fields store
the product, category, quantity, whole-dollar unit price, and month number.
`execute()` runs one SQL statement; `executemany()` binds each prepared
tuple to the `?` placeholders. `commit()` completes the insert transaction.


In [1]:
import sqlite3

import numpy as np
import pandas as pd

connection = sqlite3.connect(":memory:")

connection.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price INTEGER NOT NULL,
    month INTEGER NOT NULL
)
""")

sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]

connection.executemany("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", sales_rows)
connection.commit()

print("Prepared sales:")
for row in connection.execute("SELECT * FROM sales ORDER BY sale_id"):
    print(row)


Prepared sales:
(1, 'Mouse', 'Accessory', 3, 25, 1)
(2, 'Keyboard', 'Accessory', 2, 45, 1)
(3, 'Monitor', 'Display', 1, 180, 2)
(4, 'Laptop Stand', 'Accessory', 4, 30, 2)
(5, 'Tablet', 'Device', 2, 220, 3)


There are five sales, numbered 1 through 5. Mouse, Keyboard, and
Laptop Stand belong to Accessory; Monitor belongs to Display,
and Tablet belongs to Device.


## Convert a Numeric Result Column

SQL calculates revenue, pandas receives the named column, and `.to_numpy()`
exposes its values as an array. The array no longer carries pandas labels,
so keep the query's record order and the meaning of its values visible.


In [2]:
revenue_df = pd.read_sql_query("""
SELECT product_name, quantity * unit_price AS revenue
FROM sales
ORDER BY sale_id
""", connection)
revenue_array = revenue_df["revenue"].to_numpy()

print("Products in array order:", revenue_df["product_name"].tolist())
print("Revenue array (dollars):", revenue_array)
print("Mean revenue per sale:", np.mean(revenue_array))
print("Maximum sale revenue:", np.max(revenue_array))
print("Total revenue:", np.sum(revenue_array))


Products in array order: ['Mouse', 'Keyboard', 'Monitor', 'Laptop Stand', 'Tablet']
Revenue array (dollars): [ 75  90 180 120 440]
Mean revenue per sale: 181.0
Maximum sale revenue: 440
Total revenue: 905


The array is `[75, 90, 180, 120, 440]`. Mean revenue per sale is 181.0, maximum revenue is 440, and total revenue is 905 dollars.


## Preserve Time Order before Comparing Changes

Group by month and sort by month before converting the totals.
The prepared records cover consecutive months 1, 2, and 3 within one year.
Without that ordering, adjacent array positions would not reliably mean
adjacent months.


In [3]:
monthly_df = pd.read_sql_query("""
SELECT month, SUM(quantity * unit_price) AS total_revenue
FROM sales
GROUP BY month
ORDER BY month
""", connection)
monthly_revenue = monthly_df["total_revenue"].to_numpy()

print("Monthly revenue table:")
print(monthly_df)
print("Monthly revenue array:", monthly_revenue)


Monthly revenue table:
   month  total_revenue
0      1            165
1      2            300
2      3            440
Monthly revenue array: [165 300 440]


Months 1, 2, and 3 have revenues 165, 300, and 440 dollars. The total remains 905.


## Difference between Adjacent Totals

`np.diff(values)` subtracts each value from the next one. With three
months it produces two changes, each associated with the later month.


In [4]:
revenue_changes = np.diff(monthly_revenue)

print("Monthly changes (dollars):", revenue_changes)
for later_month, change in zip(monthly_df["month"].to_numpy()[1:], revenue_changes):
    print(f"Change ending in month {later_month}: {change} dollars")
print("Detail and monthly totals agree:", revenue_array.sum() == monthly_revenue.sum())


Monthly changes (dollars): [135 140]
Change ending in month 2: 135 dollars
Change ending in month 3: 140 dollars
Detail and monthly totals agree: True


The changes are 135 and 140 dollars, so the later increase is 5 dollars larger. These are absolute changes, not percentage growth rates. Detail and monthly totals agree.


## Close the Connection

`.close()` releases the database connection. This database lives only
in memory, so its tables disappear when the connection closes.


In [5]:
connection.close()
print("Connection closed.")


Connection closed.


Keep labels and ordering clear when moving from SQL to pandas to NumPy. Interpret `diff` as differences between adjacent values in that explicit order.

[NumPy differences](https://numpy.org/doc/stable/reference/generated/numpy.diff.html)
